# Table 4, Table 12: Ablation Study & Look-back Window Robustness

Reproduces the ablation study (component removal) and look-back window robustness tables from per-variant prediction files under `./result/ablation`.

- `ab1` = w/o Long-term horizon (month branch removed, `DeepHAR_ab1`)
- `ab2` = w/o Mid-term horizon (week branch removed, `DeepHAR_ab2`)
- `ab3` = w/o Temporal Attention Pooling (mean pooling instead of attention, `DeepHAR_ab3`)
- `ab4` = w/o Residual Path (no d_f + c_o skip connection, `DeepHAR_ab4`)
- `ab5` = w/o Shock Decomposition (no d_f - c_o deviation term, `DeepHAR_ab5`)
- `ab6` = `DeepHAR_H(configs, 4, 21)`
- `ab7` = `DeepHAR_H(configs, 6, 23)`
- `ab8` = `DeepHAR_H(configs, 3, 11)`
- `ab9` = `DeepHAR_H(configs, 10, 44)`
- Full DeepHAR (default (5,22) window) is reused from `result/pred` - it is both the Table 4 "DeepHAR" row and the Table 12 "Default (5, 22)" row.

## Required inputs
- `result/date.npy`, `result/target_{symbol}_results.npy` - shared across all tables.
- `result/pred/Seed{1..5}_DeepHAR_{symbol}_results.npy` - full-model baseline (already produced by `performance_analysis.ipynb`).
- `result/ablation/Seed{1..5}_ab{1-9}_{symbol}_results.npy` - 9 ablation variants x 5 seeds x 3 symbols = 135 files.

## Outputs (this notebook)
| File | Maps to |
|---|---|
| `result/Table4.csv` | Table 4 - Ablation study (component removal) |
| `result/Table12.csv` | Table 12 - Robustness to look-back window length |

In [1]:
import os

os.makedirs('result', exist_ok=True)

In [2]:
import pandas as pd
import numpy as np

def MSE(prediction_samples, labels) :
    N = labels.shape[0]
    prediction_samples = prediction_samples.reshape(-1)
    labels = labels.reshape(-1)
    return np.sum(np.square(prediction_samples - labels)) / N

def QLIKE(prediction_samples, labels) :
    N = labels.shape[0]
    prediction_samples = prediction_samples.reshape(-1)
    labels = labels.reshape(-1)
    return np.sum(labels/ prediction_samples -  np.log(labels/ prediction_samples) - 1)/ N

symbols = ['SPY', 'DIA', 'QQQ']
symbol_order = ['SPY', 'DIA', 'QQQ']
loss_metrics = {'MSE': MSE, 'QLIKE': QLIKE}

In [3]:
# ============================================================
# Load ablation-variant predictions from ./result/ablation
# + full-model DeepHAR baseline from ./result/pred (Table 4 "DeepHAR" row / Table 12 "Default" row)
# + target from ./result
# -> feeds Table 4 and Table 12 below
# ============================================================
seed_list = [1, 2, 3, 4, 5]   # files are saved as Seed1_ ~ Seed5_
ablation_ids = [1, 2, 3, 4, 5, 6, 7, 8, 9]   # ab1-5: Table 4, ab6-9 (orig ab12-15): Table 12

results_list = []
for seed in seed_list:
    for symbol in symbols:
        date_index = np.load('result/date.npy')

        # ablation variants
        for ab_id in ablation_ids:
            model = f'ab{ab_id}'
            values = np.load(f'./result/ablation/Seed{seed}_{model}_{symbol}_results.npy')
            results_list.append(pd.DataFrame({
                'date': date_index, 'pred': values.reshape(-1),
                'symbol': symbol, 'model': model, 'random_seed': seed
            }))

        # full DeepHAR baseline (reused from result/pred)
        values = np.load(f'./result/pred/Seed{seed}_DeepHAR_{symbol}_results.npy')
        results_list.append(pd.DataFrame({
            'date': date_index, 'pred': values.reshape(-1),
            'symbol': symbol, 'model': 'DeepHAR', 'random_seed': seed
        }))

    # target, shared across seeds
    for symbol in symbols:
        values = np.load(f'./result/target_{symbol}_results.npy')
        results_list.append(pd.DataFrame({
            'date': np.load('result/date.npy'), 'pred': values.reshape(-1),
            'symbol': symbol, 'model': 'target', 'random_seed': seed
        }))

results = pd.concat(results_list, axis=0, ignore_index=True)
print(f"Loaded {len(results)} rows "
      f"({len(ablation_ids)} ablation variants + DeepHAR + target, x {len(seed_list)} seeds x {len(symbols)} symbols)")

Loaded 248820 rows (9 ablation variants + DeepHAR + target, x 5 seeds x 3 symbols)


## Table 4 - Ablation study (component removal)

In [4]:
model_order = ['ab1', 'ab2', 'ab3', 'ab4', 'ab5', 'DeepHAR']
rename_map = {'ab1': 'w/o Long-term horizon', 'ab2': 'w/o Mid-term horizon', 'ab3': 'w/o Temporal Attention Pooling', 'ab4': 'w/o Residual Path', 'ab5': 'w/o Shock Decomposition', 'DeepHAR': 'DeepHAR'}

all_data_list = []
for seed, seed_df in results.groupby('random_seed'):
    for symbol in symbols:
        target_vals = seed_df.query('symbol == @symbol and model == "target"').sort_values('date')['pred'].values
        for model in model_order:
            pred_df = seed_df.query('symbol == @symbol and model == @model').sort_values('date')
            pred_vals = pred_df['pred'].values
            if len(pred_vals) == len(target_vals):
                for loss_name, loss_fn in loss_metrics.items():
                    all_data_list.append({
                        'seed': seed, 'symbol': symbol, 'model': model,
                        'loss': loss_name, 'performance': loss_fn(pred_vals, target_vals)
                    })

results_all = pd.DataFrame(all_data_list)
stats = results_all.groupby(['symbol', 'model', 'loss'])['performance'].agg(['mean', 'std']).reset_index()
stats['display'] = stats.apply(lambda r: f"{r['mean']:.3f} $\\pm$ {r['std']:.3f}", axis=1)

pivot_df = stats.pivot_table(index='model', columns=['symbol', 'loss'], values='display', aggfunc='first')
pivot_df = pivot_df.reindex(index=model_order)
pivot_df = pivot_df.reindex(columns=symbol_order, level='symbol')
pivot_df = pivot_df.reindex(columns=['MSE', 'QLIKE'], level='loss')
pivot_df = pivot_df.rename(index=rename_map)

pivot_df.T.to_csv("result/Table4.csv")
print(pivot_df)

symbol                                        SPY                     \
loss                                          MSE              QLIKE   
model                                                                  
w/o Long-term horizon           2.791 $\pm$ 0.041  0.224 $\pm$ 0.002   
w/o Mid-term horizon            2.912 $\pm$ 0.041  0.223 $\pm$ 0.002   
w/o Temporal Attention Pooling  2.899 $\pm$ 0.041  0.220 $\pm$ 0.000   
w/o Residual Path               2.763 $\pm$ 0.131  0.223 $\pm$ 0.002   
w/o Shock Decomposition         3.030 $\pm$ 0.315  0.219 $\pm$ 0.002   
DeepHAR                         2.756 $\pm$ 0.055  0.219 $\pm$ 0.002   

symbol                                        DIA                     \
loss                                          MSE              QLIKE   
model                                                                  
w/o Long-term horizon           2.511 $\pm$ 0.053  0.173 $\pm$ 0.002   
w/o Mid-term horizon            2.589 $\pm$ 0.017  0.172 $\pm$ 

## Table 12 - Robustness to look-back window length

In [5]:
model_order = ['ab8', 'ab6', 'ab7', 'ab9', 'DeepHAR']
rename_map = {'ab8': '(3, 11)', 'ab6': '(4, 21)', 'ab7': '(6, 23)', 'ab9': '(10, 44)', 'DeepHAR': 'Default (5, 22)'}

all_data_list = []
for seed, seed_df in results.groupby('random_seed'):
    for symbol in symbols:
        target_vals = seed_df.query('symbol == @symbol and model == "target"').sort_values('date')['pred'].values
        for model in model_order:
            pred_df = seed_df.query('symbol == @symbol and model == @model').sort_values('date')
            pred_vals = pred_df['pred'].values
            if len(pred_vals) == len(target_vals):
                for loss_name, loss_fn in loss_metrics.items():
                    all_data_list.append({
                        'seed': seed, 'symbol': symbol, 'model': model,
                        'loss': loss_name, 'performance': loss_fn(pred_vals, target_vals)
                    })

results_all = pd.DataFrame(all_data_list)
stats = results_all.groupby(['symbol', 'model', 'loss'])['performance'].agg(['mean', 'std']).reset_index()
stats['display'] = stats.apply(lambda r: f"{r['mean']:.3f} $\\pm$ {r['std']:.3f}", axis=1)

pivot_df = stats.pivot_table(index='model', columns=['symbol', 'loss'], values='display', aggfunc='first')
pivot_df = pivot_df.reindex(index=model_order)
pivot_df = pivot_df.reindex(columns=symbol_order, level='symbol')
pivot_df = pivot_df.reindex(columns=['MSE', 'QLIKE'], level='loss')
pivot_df = pivot_df.rename(index=rename_map)

pivot_df.T.to_csv("result/Table12.csv")
print(pivot_df)

symbol                         SPY                                   DIA  \
loss                           MSE              QLIKE                MSE   
model                                                                      
(3, 11)          2.864 $\pm$ 0.169  0.222 $\pm$ 0.001  2.611 $\pm$ 0.262   
(4, 21)          2.971 $\pm$ 0.252  0.220 $\pm$ 0.003  2.733 $\pm$ 0.354   
(6, 23)          2.983 $\pm$ 0.208  0.222 $\pm$ 0.002  2.726 $\pm$ 0.303   
(10, 44)         2.842 $\pm$ 0.138  0.221 $\pm$ 0.002  2.562 $\pm$ 0.172   
Default (5, 22)  2.756 $\pm$ 0.055  0.219 $\pm$ 0.002  2.468 $\pm$ 0.064   

symbol                                            QQQ                     
loss                         QLIKE                MSE              QLIKE  
model                                                                     
(3, 11)          0.170 $\pm$ 0.002  3.819 $\pm$ 0.195  0.197 $\pm$ 0.002  
(4, 21)          0.168 $\pm$ 0.004  3.918 $\pm$ 0.290  0.197 $\pm$ 0.003  
(6, 23)         